<a href="https://colab.research.google.com/github/abdabdelli172-collab/TP1_MODELE_DIABETE/blob/main/TP1_Modele_Diabete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import pandas as pd
import numpy as np

colonnes = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

# Charger la base originale
df_original = pd.read_csv(
    "pima-indians-diabetes.csv",
    header=None,
    names=colonnes
)

# Garder une copie de sécurité
df = df_original.copy()

print("Dataset original :", df_original.shape)
print(df_original["Outcome"].value_counts())


Dataset original : (768, 9)
Outcome
0    500
1    268
Name: count, dtype: int64


In [8]:

# Créer une copie pour l'amélioration
df_ameliore = df_original.copy()

print("Base originale :", df_original.shape)
print("Copie pour amélioration :", df_ameliore.shape)


Base originale : (768, 9)
Copie pour amélioration : (768, 9)


In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

from google.colab import files

# Noms des colonnes
colonnes = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

# Charger le dataset original
uploaded = files.upload()

nom = list(uploaded.keys())[0]

df_original = pd.read_csv(
    nom,
    header=None,
    names=colonnes
)

# Créer une copie de travail
df_ameliore = df_original.copy(deep=True)

print("Dataset original :", df_original.shape)
print("Copie de travail :", df_ameliore.shape)

print("\nDistribution originale :")
print(df_original["Outcome"].value_counts())


Saving pima-indians-diabetes.csv to pima-indians-diabetes (1).csv
Dataset original : (768, 9)
Copie de travail : (768, 9)

Distribution originale :
Outcome
0    500
1    268
Name: count, dtype: int64


In [10]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Séparer les variables
X = df_ameliore.drop("Outcome", axis=1).copy()
y = df_ameliore["Outcome"].copy()

# Remplacer les zéros considérés comme valeurs manquantes
colonnes_zero = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[colonnes_zero] = X[colonnes_zero].replace(0, np.nan)

# Traitement des valeurs manquantes
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Équilibrage avec SMOTE
smote = SMOTE(
    sampling_strategy={0: 1000, 1: 1000},
    random_state=42
)

X_resampled, y_resampled = smote.fit_resample(
    X_scaled, y
)

# Retour à l'échelle originale
X_final = scaler.inverse_transform(X_resampled)

# Création du nouveau dataset
df_ameliore = pd.DataFrame(
    X_final,
    columns=X.columns
)

df_ameliore["Outcome"] = y_resampled

print("Dataset amélioré :", df_ameliore.shape)
print("\nDistribution après SMOTE :")
print(df_ameliore["Outcome"].value_counts())


Dataset amélioré : (2000, 9)

Distribution après SMOTE :
Outcome
1    1000
0    1000
Name: count, dtype: int64


In [11]:

# Sauvegarder le dataset amélioré
df_ameliore.to_csv(
    "DIACARE_dataset_ameliore_2000.csv",
    index=False
)

print("Dataset sauvegardé avec succès !")
print("Nombre d'exemples :", len(df_ameliore))
print("Nombre de colonnes :", len(df_ameliore.columns))


Dataset sauvegardé avec succès !
Nombre d'exemples : 2000
Nombre de colonnes : 9


In [12]:

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Variables explicatives et cible
X = df_original.drop("Outcome", axis=1)
y = df_original["Outcome"]

# Division des données
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Création du pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

# Entraînement
pipeline.fit(X_train, y_train)

print("Entraînement terminé avec succès !")


Entraînement terminé avec succès !


In [19]:
!pip install xgboost -q

In [22]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# النماذج
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

et = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(C=0.5, max_iter=2000))
])

# دمج النماذج
voting = VotingClassifier(
    estimators=[
        ("rf", rf),
        ("et", et),
        ("xgb", xgb),
        ("lr", lr)
    ],
    voting="soft",
    weights=[2, 2, 1, 1]
)

# المعالجة والتدريب
modele = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("voting", voting)
])

modele.fit(X_train, y_train)

# التقييم
y_pred = modele.predict(X_test)

print("Accuracy test:",
      round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nRapport de classification:")
print(classification_report(y_test, y_pred))

print("\nMatrice de confusion:")
print(confusion_matrix(y_test, y_pred))

Accuracy test: 72.08 %

Rapport de classification:
              precision    recall  f1-score   support

           0       0.75      0.85      0.80       100
           1       0.63      0.48      0.55        54

    accuracy                           0.72       154
   macro avg       0.69      0.67      0.67       154
weighted avg       0.71      0.72      0.71       154


Matrice de confusion:
[[85 15]
 [28 26]]


In [23]:
!pip install tensorflow -q

In [25]:
print("Nombre total :", len(df_original))

print("\nRépartition des classes :")
print(df_original["Outcome"].value_counts())

print("\nMeilleure accuracy de validation :",
      max(historique.history["val_accuracy"]) * 100)

print("\nDernière accuracy de validation :",
      historique.history["val_accuracy"][-1] * 100)

print("\nAccuracy de test :",
      accuracy * 100)

Nombre total : 768

Répartition des classes :
Outcome
0    500
1    268
Name: count, dtype: int64

Meilleure accuracy de validation : 82.92682766914368

Dernière accuracy de validation : 81.30081295967102

Accuracy de test : 72.07792401313782


In [28]:
!pip install catboost -q

In [34]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# البيانات الأصلية
X = df_original.drop("Outcome", axis=1).copy()
y = df_original["Outcome"].copy()

# معالجة القيم الصفرية
colonnes_zero = ["Glucose", "BloodPressure",
                 "SkinThickness", "Insulin", "BMI"]

X[colonnes_zero] = X[colonnes_zero].replace(0, np.nan)

# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# إنشاء النموذج
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ))
])

# الإعدادات
parametres = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

recherche = RandomizedSearchCV(
    pipeline,
    param_distributions=parametres,
    n_iter=40,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

# التدريب
recherche.fit(X_train, y_train)

# أفضل نموذج
modele = recherche.best_estimator_

# التقييم النهائي
y_pred = modele.predict(X_test)

print("Meilleurs paramètres:")
print(recherche.best_params_)

print("\nAccuracy validation croisée:",
      round(recherche.best_score_ * 100, 2), "%")

print("\nAccuracy test:",
      round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nRapport de classification:")
print(classification_report(y_test, y_pred))

print("\nMatrice de confusion:")
print(confusion_matrix(y_test, y_pred))

Meilleurs paramètres:
{'model__subsample': 0.7, 'model__n_estimators': 100, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.7}

Accuracy validation croisée: 77.85 %

Accuracy test: 76.62 %

Rapport de classification:
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       100
           1       0.70      0.59      0.64        54

    accuracy                           0.77       154
   macro avg       0.75      0.73      0.73       154
weighted avg       0.76      0.77      0.76       154


Matrice de confusion:
[[86 14]
 [22 32]]


In [35]:
# الاحتفاظ بالنموذج الأول
meilleur_modele = modele_dl_initial

# حفظ النموذج
meilleur_modele.save("DIACARE_best_model.keras")

# حفظ أدوات المعالجة
import joblib

joblib.dump(imputer_dl, "DIACARE_imputer.pkl")
joblib.dump(scaler_dl, "DIACARE_scaler.pkl")

print("Meilleur modèle sauvegardé avec succès.")

Meilleur modèle sauvegardé avec succès.


In [37]:
import time

patient = [2, 150, 80, 25, 100, 32.0, 0.5, 45]

# تحويل بيانات المريض
colonnes = [
    "Pregnancies", "Glucose", "BloodPressure",
    "SkinThickness", "Insulin", "BMI",
    "DiabetesPedigreeFunction", "Age"
]

donnees = pd.DataFrame([patient], columns=colonnes)

# قياس وقت 1000 تنبؤ
debut = time.perf_counter()

for i in range(1000):
    modele.predict(donnees)

fin = time.perf_counter()

temps_moyen = (fin - debut) / 1000

print("Temps moyen de prédiction :",
      round(temps_moyen * 1000, 4), "ms")

Temps moyen de prédiction : 4.5236 ms


In [38]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = modele.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7662337662337663

Rapport de classification :
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       100
           1       0.70      0.59      0.64        54

    accuracy                           0.77       154
   macro avg       0.75      0.73      0.73       154
weighted avg       0.76      0.77      0.76       154


Matrice de confusion :
[[86 14]
 [22 32]]


In [39]:
import joblib

# Sauvegarder le modèle complet
joblib.dump(modele, "DIACARE_XGBoost.pkl")

print("Modèle sauvegardé avec succès !")

Modèle sauvegardé avec succès !


In [40]:
from google.colab import files

files.download("DIACARE_XGBoost.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
def predire(entree):

    colonnes = [
        "Pregnancies",
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI",
        "DiabetesPedigreeFunction",
        "Age"
    ]

    donnees = pd.DataFrame([entree], columns=colonnes)

    colonnes_zero = [
        "Glucose", "BloodPressure",
        "SkinThickness", "Insulin", "BMI"
    ]

    donnees[colonnes_zero] = donnees[colonnes_zero].replace(0, np.nan)

    prediction = modele.predict(donnees)[0]
    probabilite = modele.predict_proba(donnees)[0][1] * 100

    resultat = "Risque de diabète" if prediction == 1 else "Risque faible de diabète"

    return resultat, round(float(probabilite), 2)


# Exemple de patient
patient = [2, 150, 80, 25, 100, 32.0, 0.5, 45]

resultat, probabilite = predire(patient)

print("Résultat :", resultat)
print("Probabilité estimée :", probabilite, "%")

Résultat : Risque de diabète
Probabilité estimée : 53.32 %


In [42]:
patient2 = [1, 90, 70, 20, 80, 24.5, 0.3, 30]

resultat, probabilite = predire(patient2)

print("Résultat :", resultat)
print("Probabilité estimée :", probabilite, "%")

Résultat : Risque faible de diabète
Probabilité estimée : 1.26 %


In [43]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = modele.predict(X_test)

print("Accuracy Test :",
      round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nClassification Report :")
print(classification_report(y_test, y_pred))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Accuracy Test : 76.62 %

Classification Report :
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       100
           1       0.70      0.59      0.64        54

    accuracy                           0.77       154
   macro avg       0.75      0.73      0.73       154
weighted avg       0.76      0.77      0.76       154


Matrice de confusion :
[[86 14]
 [22 32]]


In [44]:
import joblib

joblib.dump(modele, "DIACARE_XGBoost.pkl")

print("Modèle sauvegardé avec succès !")

Modèle sauvegardé avec succès !


In [45]:
from google.colab import files

files.download("DIACARE_XGBoost.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import joblib

modele_charge = joblib.load("DIACARE_XGBoost.pkl")

patient = pd.DataFrame([[
    2, 150, 80, 25, 100, 32.0, 0.5, 45
]], columns=X.columns)

prediction = modele_charge.predict(patient)[0]
probabilite = modele_charge.predict_proba(patient)[0][1] * 100

print("Prediction :", prediction)
print("Probabilité :", round(float(probabilite), 2), "%")

Prediction : 1
Probabilité : 53.32 %
